# VeriForgot — Notebook 04: Oracle Detection (Genuine vs Fake)

Tests oracle detection on:
- 10 genuinely unlearned models (varying GA epochs)
- 10 fake models (Gaussian noise perturbation only)

Produces Table 4 and Figure 3 (fig3_oracle.png) from the CRBL 2026 paper.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import torchvision.models as models
import numpy as np, pickle
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Subset

SEED=42; TAU=0.57; torch.manual_seed(SEED); np.random.seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Oracle verdict function
def oracle_verdict(model, forget_loader, nm_loader):
    model.eval(); mc=[]; nc=[]
    with torch.no_grad():
        for x,y in forget_loader:
            x,y=x.to(device),y.to(device)
            p=torch.softmax(model(x),dim=1)
            mc.extend(p[torch.arange(len(y)),y].cpu().numpy())
        for x,y in nm_loader:
            x,y=x.to(device),y.to(device)
            p=torch.softmax(model(x),dim=1)
            nc.extend(p[torch.arange(len(y)),y].cpu().numpy())
    mc,nc=np.array(mc),np.array(nc)
    labels=np.concatenate([np.ones(len(mc)),np.zeros(len(nc))])
    auc=roc_auc_score(labels,np.concatenate([mc,nc]))
    return auc < TAU, float(auc)

# Run 10 genuine + 10 fake
# (full code in paper supplementary / Final Notebook on Kaggle)